# Rodrigo AI — synthetic-baseline

Job determinístico para validar o contrato. Não mede voz, qualidade ou latência do aplicativo e não usa rede, GPU, dados pessoais ou chaves. No Colab, defina `RODRIGO_AI_COMMIT` com o commit de origem antes de executar.

In [ ]:
import datetime, hashlib, json, os, platform, sys, time
from pathlib import Path

RUN_ID = os.environ.get('RODRIGO_AI_RUN_ID', 'synthetic-local-001')
COMMIT = os.environ.get('RODRIGO_AI_COMMIT')
if not COMMIT or not all(c in '0123456789abcdef' for c in COMMIT.lower()) or len(COMMIT) < 7:
    raise ValueError('Defina RODRIGO_AI_COMMIT com git rev-parse HEAD; não invente a revisão')
OUT = Path(os.environ.get('RODRIGO_AI_OUTPUT', '/content/rodrigo-ai-colab-output'))
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
job_sha256 = hashlib.sha256(Path('synthetic-baseline.ipynb').read_bytes()).hexdigest() if Path('synthetic-baseline.ipynb').exists() else os.environ.get('RODRIGO_AI_JOB_SHA256')
if not job_sha256:
    raise ValueError('Forneça RODRIGO_AI_JOB_SHA256 quando o notebook estiver aberto fora do checkout')
try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    gpu_available = bool(torch.cuda.is_available())
except Exception:
    gpu_name, gpu_available = None, False
result = {
    'schema_version': 1, 'job': 'synthetic-baseline', 'run_id': RUN_ID,
    'status': 'success', 'metrics': {'synthetic_items': 100.0, 'synthetic_checksum': 5050.0},
    'gpu': {'name': gpu_name, 'available': gpu_available},
    'duration_seconds': round(time.perf_counter() - started, 6), 'commit': COMMIT.lower(),
    'source': {'job_sha256': job_sha256, 'dependencies': {'python': platform.python_version()}, 'dirty': False},
    'created_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
tmp = OUT / '.result.json.tmp'
tmp.write_text(json.dumps(result, indent=2, sort_keys=True) + '\n', encoding='utf-8')
tmp.replace(OUT / 'result.json')
print(json.dumps(result, indent=2, sort_keys=True))